# 02D Quick — 34-circle M2 diagnostic BEFORE spatial CV

**Purpose:** reuse the new 02C M2 checkpoint (27 development circles and original seven independent holdouts), predict **all 34 Nadine circles**, and inspect predicted versus observed fractional Musanga cover, by site. **No training and no illumination experiments.**

**Crucial:** the 27 development circles were available during training. Their performance is *in-sample/apparent*, **not independent validation**. Only the seven holdouts estimate out-of-sample performance. Full independent 34-circle assessment still requires three-fold CV.

Read-only source model: `Final/Outputs/V3/02C_nadine34_legacy7/M2_model.pt`. Outputs: `Final/Outputs/V3/02D_quick34_diagnostic/`. Historical 02D remains unchanged.


## 01 — Environment / NADINE34 dataset

In [ ]:
%pip -q install rasterio geopandas pyogrio shapely pandas matplotlib scikit-learn
from pathlib import Path
import json, warnings, math, random, time
import numpy as np, pandas as pd, geopandas as gpd, rasterio, matplotlib.pyplot as plt
import torch, torch.nn as nn, torch.nn.functional as F
from rasterio.windows import Window
from rasterio.features import rasterize
from shapely.geometry import box
from shapely.ops import unary_union
from shapely import make_valid
from torch.utils.data import DataLoader, Dataset
from sklearn.metrics import average_precision_score, roc_auc_score
from IPython.display import display
from google.colab import drive
if not Path('/content/drive/MyDrive').exists():drive.mount('/content/drive')
ROOT=next((p for p in [Path('/content/drive/MyDrive/Colab Notebooks/Musanga/Final'),
                         Path('/content/drive/MyDrive/Musanga/Final')] if p.is_dir()),None)
assert ROOT is not None
OUT=ROOT/'Outputs/V3/02D_quick34_diagnostic';OUT.mkdir(parents=True,exist_ok=True)
MODEL_SOURCE=ROOT/'Outputs/V3/02C_nadine34_legacy7/M2_model.pt'
REF=ROOT/'Outputs/V3/01_reference_data_nadine34'
A=json.loads((ROOT/'Outputs/V3/00_input_audit/00_input_manifest.json').read_text())
assert A.get('schema_version')=='v3_input_audit_5_nadine_circles'
assert A['exhaustive_annotations'].get('all_annotation_polygons_musanga') is True
assert int(A['exhaustive_annotations']['verified_exhaustive_circle_count'])==34
mosaics={m['file']:m for m in A['mosaics']}
splits=pd.read_csv(REF/'01_cv_window_manifest.csv')
links=pd.read_csv(REF/'01_circle_spatial_folds.csv')
repairs=pd.read_csv(REF/'01_geometry_repairs.csv')
assert links.circle_id.nunique()==34
assert links.groupby('site').circle_id.nunique().to_dict()=={'Baego':7,'MOS':15,'Yangambi':7,'Yoko':5}
assert links.circle_id.is_unique
FOLD=0;SEED=20261016;CROP=256;EPOCHS=25;BATCH=8
MAX_PARTIAL_PER_MOSAIC=250
MAX_EXHAUSTIVE_PER_SITE=260
HOLDOUT_COUNTS={'Baego':1,'MOS':2,'Yangambi':2,'Yoko':2}
assert torch.cuda.is_available(),'Select Runtime > Change runtime type > GPU for inference.'
device='cuda';torch.manual_seed(SEED);np.random.seed(SEED);random.seed(SEED)
print('NADINE 34 circles; legacy seven-circle independent holdout; new output isolated')
print('GPU',torch.cuda.get_device_name(0),'| fold',FOLD)
print('Train/val candidates:',splits[splits.cv_fold==FOLD].split.value_counts().to_dict())
print('Files:',ROOT,'| output:',OUT)




assert int(A['exhaustive_annotations']['musanga_polygons'])==1368
print('QUICK34: no model training, checkpoint expected:',MODEL_SOURCE)


## 02 — Geometry and M2 architecture (unchanged from 02C)

In [ ]:
def pieces(g):
    if g.geom_type in ('Polygon','MultiPolygon'):return [g]
    if hasattr(g,'geoms'):return [p for q in g.geoms for p in pieces(q)]
    return []
def polygons(path):
    src=gpd.read_file(path);inds=[];shapes=[]
    for i,g in enumerate(src.geometry):
        if g is None or g.is_empty:continue
        if not g.is_valid:
            try:
                fixed=unary_union(pieces(make_valid(g)))
                change=abs(fixed.area-g.area)/max(fixed.area,g.area,1e-12)
                if change>(.01 if Path(path) in (Path(A['exhaustive_annotations']['complete_roi_boundaries']),Path(A['exhaustive_annotations']['positive_polygons']),Path(A['roi'])) else .05) or not fixed.is_valid:continue
                g=fixed
            except Exception:continue
        inds.append(i);shapes.append(g)
    x=src.iloc[inds].copy().reset_index(drop=True)
    x.geometry=shapes
    x['original_row']=inds
    return x
roi=polygons(A['roi'])
circles=polygons(A['exhaustive_annotations']['complete_roi_boundaries'])
positive=polygons(A['exhaustive_annotations']['positive_polygons'])
assert len(circles)==34 and len(positive)==1368,(
    'Unexpected reference geometry count; do not train using a partial dataset')
circle_ids=set(links.circle_id.astype(int))
circles=circles[circles.original_row.isin(circle_ids)].copy()
tree={}
for site,path in A['shared_annotations'].items():
    t=polygons(path)
    cls=pd.to_numeric(t.Class_ID,errors='coerce')
    t['class01']=np.where(cls.eq(1),1,np.where(cls.notna(),0,-1))
    tree[site]=t
geo={}
for name,m in mosaics.items():
    with rasterio.open(m['path']) as src:
        c=circles.to_crs(src.crs).copy()
        c['fold']=c.original_row.map(links.set_index('circle_id').fold)
        c['owner']=c.original_row.map(links.set_index('circle_id').mosaic)
        geo[name]=dict(site=m['site'],path=m['path'],transform=src.transform,
            crs=src.crs,roi=roi.to_crs(src.crs),
            circles=c,positive=positive.to_crs(src.crs),trees=tree[m['site']].to_crs(src.crs))



In [ ]:
class TinyUNet(nn.Module):
    def __init__(self,base=16):
        super().__init__()
        def block(a,b):
            return nn.Sequential(nn.Conv2d(a,b,3,padding=1),nn.BatchNorm2d(b),nn.ReLU(inplace=True),
                nn.Conv2d(b,b,3,padding=1),nn.BatchNorm2d(b),nn.ReLU(inplace=True))
        self.e1=block(3,base);self.e2=block(base,base*2)
        self.bridge=block(base*2,base*4)
        self.d2=block(base*4+base*2,base*2)
        self.d1=block(base*2+base,base)
        self.final=nn.Conv2d(base,1,1)
    def forward(self,x):
        e1=self.e1(x);e2=self.e2(F.max_pool2d(e1,2))
        b=self.bridge(F.max_pool2d(e2,2))
        x=self.d2(torch.cat([F.interpolate(b,size=e2.shape[-2:],mode='bilinear',align_corners=False),e2],1))
        x=self.d1(torch.cat([F.interpolate(x,size=e1.shape[-2:],mode='bilinear',align_corners=False),e1],1))
        return self.final(x)[:,0]
class CropData(Dataset):
    def __init__(self,rows,augment=False):self.rows=rows;self.augment=augment
    def __len__(self):return len(self.rows)
    def __getitem__(self,i):
        d=self.rows[i];image=d['rgb'];y=d['y'];w=d['w']
        if self.augment:
            rng=np.random.default_rng(SEED+i+int(torch.randint(0,1000000,(1,)).item()))
            k=int(rng.integers(4));image=np.rot90(image,k);y=np.rot90(y,k);w=np.rot90(w,k)
            if rng.random()<.5:image=image[:,::-1];y=y[:,::-1];w=w[:,::-1]
        x=torch.tensor(np.ascontiguousarray(image.transpose(2,0,1)),dtype=torch.float32)
        return x,torch.tensor(np.ascontiguousarray(y)),torch.tensor(np.ascontiguousarray(w))
def masked_loss(logits,y,w):
    loss=F.binary_cross_entropy_with_logits(logits,y,reduction='none')
    return (loss*w).sum()/w.sum().clamp_min(1.)
print('Tiny U-Net ready (native 256×256 RGB crops).')




## 03 — Historical seven independent holdouts / 27 development circles

In [ ]:
SPLIT_SOURCE=ROOT/'Outputs/V3/02C_nadine34_legacy7/02C_frozen_circle_split.csv'
assert SPLIT_SOURCE.exists(),'Run 02C Block 04 first; split file missing.'
split_df=pd.read_csv(SPLIT_SOURCE)
assert {'circle_id','site','mosaic','split'}.issubset(split_df)
assert len(split_df)==34 and split_df.circle_id.is_unique
assert set(split_df.circle_id.astype(int))==set(links.circle_id.astype(int))
HOLDOUT=set(split_df.loc[split_df.split=='holdout','circle_id'].astype(int))
DEV=set(split_df.loc[split_df.split=='development','circle_id'].astype(int))
assert HOLDOUT=={2,5,8,13,14,17,20} and len(DEV)==27
BLOCKED={}  # training-only structure, not used for evaluating the 34 circles
print('DEV apparent fit:',len(DEV),'| independent holdout:',sorted(HOLDOUT))


## 04 — Raster reads / reference masks; no training cache constructed

In [ ]:
from rasterio.windows import Window
from torch.utils.data import Dataset,DataLoader
candidate=pd.read_csv(REF/'01_candidate_windows.csv')
all_circle_buffers={}
for name,d in geo.items():
    cs=circles.to_crs(d['crs'])
    all_circle_buffers[name]=[g.buffer(15.) for g in cs.geometry]
    d['dev_circles']=cs[cs.original_row.isin(DEV)]
    d['holdout_circles']=cs[cs.original_row.isin(HOLDOUT)]
    d['positive_local']=positive.to_crs(d['crs'])
    d['trees_local']=tree[d['site']].to_crs(d['crs'])
    d['roi_local']=roi.to_crs(d['crs'])

def burn_window(series,name,win):
    d=geo[name];bb=box(*rasterio.windows.bounds(win,d['transform']))
    gs=[g.intersection(bb) for g in series if g is not None and g.intersects(bb)]
    if not gs:return np.zeros((int(win.height),int(win.width)),bool)
    return rasterize([(g,1) for g in gs if not g.is_empty],
       out_shape=(int(win.height),int(win.width)),transform=rasterio.windows.transform(win,d['transform']),
       fill=0,dtype='uint8').astype(bool)

def rgb_for_window(name,win):
    with rasterio.open(geo[name]['path']) as ds:
        raw=ds.read([1,2,3],window=win,masked=True)
    valid=~np.any(np.ma.getmaskarray(raw),axis=0)
    a=raw.filled(0).astype('float32')
    valid &= np.any(a!=0,axis=0)&np.any(a!=255,axis=0)
    valid &= ~((a[0]==0)&(a[1]==0)&(a[2]==255)) # blue Baego nodata
    return np.clip(a/255.,0,1).transpose(1,2,0).astype('float32'),valid



## 05 — Load the *completed* Nadine34 M2 checkpoint

In [ ]:
assert MODEL_SOURCE.exists(),(
    f'No completed M2 checkpoint at {MODEL_SOURCE}. Wait for NB02C Block 06 to finish.')
model2=TinyUNet().to(device)
model2.load_state_dict(torch.load(MODEL_SOURCE,map_location=device,weights_only=True))
model2.eval()
print('LOADED existing M2 weights; ZERO training epochs in this notebook.',flush=True)


## 06 — Evaluate all 34 circles; store exact reference pixel support

In [ ]:
from scipy.special import expit,logit
from scipy.optimize import minimize
from sklearn.metrics import r2_score,mean_squared_error,mean_absolute_error
WINDOWS=pd.read_csv(REF/'01_candidate_windows.csv')
VAL_WINDOWS=WINDOWS[WINDOWS.kind=='exhaustive'].copy()
assert set(VAL_WINDOWS.circle_id.unique().astype(int))==set(links.circle_id.astype(int))
N_BINS=256
BINS=np.linspace(0,1,N_BINS+1)
BIN_CENTER=(BINS[:-1]+BINS[1:])/2

def run_all_circles(model,model_name):
    statfile=OUT/f'{model_name}_quick34_circle_raw.csv'
    histfile=OUT/f'{model_name}_quick34_histograms.npz'
    if statfile.exists() and histfile.exists():
        return pd.read_csv(statfile),dict(np.load(histfile))
    model.eval();rows=[];histograms={}
    for _,circle in split_df.iterrows():
        cid=int(circle.circle_id)
        name=circle.mosaic
        d=geo[name]
        windows=VAL_WINDOWS[(VAL_WINDOWS.circle_id==cid)&(VAL_WINDOWS.mosaic==name)]
        assert len(windows),f'No evaluation windows for circle {cid}'
        cg=circles[circles.original_row==cid].to_crs(d['crs'])
        counts=np.zeros(N_BINS,np.int64);true_counts=np.zeros(N_BINS,np.int64)
        n=0;sumy=0.;sump=0.;sum_sq=0.;tp=fp=fn=0
        rng=np.random.default_rng(SEED+cid)
        sample_truth=[];sample_probs=[]
        for rec in windows.itertuples(index=False):
            win=Window(int(rec.col),int(rec.row),int(rec.width),int(rec.height))
            mask=burn_window(cg.geometry,name,win)
            mask &= burn_window(d['roi_local'].geometry,name,win)
            rgb,valid=rgb_for_window(name,win)
            mask &= valid
            if not mask.any():continue
            reference=burn_window(d['positive_local'].geometry,name,win)
            x=torch.from_numpy(rgb.transpose(2,0,1).copy())[None].to(device)
            with torch.inference_mode():
                pred=torch.sigmoid(model(x))[0].cpu().numpy()
            y=reference[mask].astype('uint8');p=pred[mask].astype('float32')
            n+=len(y);sumy+=float(y.sum());sump+=float(p.sum())
            sum_sq+=float(np.square(p-y).sum())
            binary=p>=.5
            tp+=int(np.sum(binary&(y==1)));fp+=int(np.sum(binary&(y==0)))
            fn+=int(np.sum((~binary)&(y==1)))
            ix=np.minimum((p*N_BINS).astype(np.int32),N_BINS-1)
            counts+=np.bincount(ix,minlength=N_BINS).astype('int64')
            true_counts+=np.bincount(ix,weights=y,minlength=N_BINS).astype('int64')
            # only diagnostic AUC/AP uses pixel subsampling, with fixed per-circle cap
            take=rng.choice(len(y),min(100,len(y)),replace=False)
            sample_truth.extend(y[take].tolist());sample_probs.extend(p[take].tolist())
        assert n>0,f'No valid ROI/RGB reference pixels for circle {cid}'
        assert int(counts.sum())==n
        yt=np.asarray(sample_truth);pr=np.asarray(sample_probs)
        rows.append(dict(model=model_name,circle_id=cid,site=circle.site,split=circle.split,
                         valid_pixels=n,observed_fraction=100*sumy/n,
                         predicted_fraction=100*sump/n,
                         bias_pp=100*(sump-sumy)/n,brier=sum_sq/n,
                         f1_threshold_05=2*tp/max(1,2*tp+fp+fn),
                         auc_sample=roc_auc_score(yt,pr) if len(np.unique(yt))>1 else np.nan,
                         ap_sample=average_precision_score(yt,pr) if len(np.unique(yt))>1 else np.nan))
        histograms[str(cid)+'_count']=counts
        histograms[str(cid)+'_positive']=true_counts
        print(model_name,'circle',cid,'obs',round(100*sumy/n,1),
              'pred',round(100*sump/n,1),'valid pixels',n,flush=True)
    frame=pd.DataFrame(rows)
    assert frame.circle_id.is_unique
    assert set(frame.circle_id.astype(int))==set(links.circle_id.astype(int))
    assert len(frame)==34
    frame.to_csv(statfile,index=False)
    np.savez_compressed(histfile,**histograms)
    return frame,histograms

m2, hist2=run_all_circles(model2,'M2')
print('M2: all 34 circle predictions saved; seven are independent holdouts.',flush=True)



## 07 — Clear distinction: apparent fit versus independent validation

In [ ]:
from scipy.stats import linregress
from sklearn.metrics import r2_score,mean_squared_error,mean_absolute_error
assert len(m2)==34 and m2.circle_id.is_unique
assert m2.groupby('split').circle_id.nunique().to_dict()=={'development':27,'holdout':7}

def summarize(q):
    y=q.observed_fraction.to_numpy(float)
    p=q.predicted_fraction.to_numpy(float)
    lr=linregress(y,p) if len(q)>2 and np.std(y)>0 else None
    return pd.Series(dict(n=len(q),R2=float(r2_score(y,p)) if len(q)>1 else np.nan,
        MAE_pp=float(mean_absolute_error(y,p)),
        RMSE_pp=float(np.sqrt(mean_squared_error(y,p))),
        bias_pp=float(np.mean(p-y)),
        slope=float(lr.slope) if lr else np.nan,
        intercept_pp=float(lr.intercept) if lr else np.nan,
        brier=float(q.brier.mean())))
summary=[]
for split in ['development','holdout']:
    q=m2[m2.split==split]
    summary.append(dict(group=('APPARENT training-set fit' if split=='development'
                               else 'INDEPENDENT held-out validation'),**summarize(q).to_dict()))
summary.append(dict(group='ALL 34 descriptive (MIXED, NOT VALIDATION)',
                    **summarize(m2).to_dict()))
summary=pd.DataFrame(summary)
display(summary.round(3))
by_site=m2.groupby(['split','site']).apply(summarize,include_groups=False).reset_index()
display(by_site.round(3))
m2.to_csv(OUT/'02D_quick34_circle_predictions.csv',index=False)
summary.to_csv(OUT/'02D_quick34_overview.csv',index=False)
by_site.to_csv(OUT/'02D_quick34_site_summary.csv',index=False)

colors={'Baego':'#c27e24','MOS':'#357ebd','Yangambi':'#468c50','Yoko':'#944b99'}
fig,axs=plt.subplots(1,2,figsize=(12,5.4),sharex=True,sharey=True,constrained_layout=True)
for ax,split,title in zip(axs,['development','holdout'],
      ['27 development circles (apparent fit)','7 independent holdouts']):
    q=m2[m2.split==split]
    ax.plot([0,100],[0,100],ls='--',lw=1,color='gray')
    for row in q.itertuples(index=False):
        ax.scatter(row.observed_fraction,row.predicted_fraction,
                   color=colors[row.site],s=55,edgecolor='white',linewidth=.6)
        ax.annotate(str(row.circle_id),(row.observed_fraction,row.predicted_fraction),
                    xytext=(4,4),textcoords='offset points',fontsize=7)
    metric=summarize(q)
    ax.set(xlim=(0,100),ylim=(0,100),
           title=f'{title}\nR²={metric.R2:.3f} | MAE={metric.MAE_pp:.2f} pp',
           xlabel='Observed Musanga fraction (%)')
    ax.grid(alpha=.2)
axs[0].set_ylabel('Predicted mean Musanga probability (%)')
fig.savefig(OUT/'02D_quick34_development_vs_holdout.png',dpi=180,bbox_inches='tight')
display(fig);plt.close(fig)
print('Interpretation: all-34 metrics are NOT spatial CV. Compare seven-holdout scores to historical 02C.')


## 08 — Audit provenance and stop before any model selection

In [ ]:
contract=dict(schema='musanga_v3_02D_quick34_m2_diagnostic',
    input_manifest=str(ROOT/'Outputs/V3/00_input_audit/00_input_manifest.json'),
    reference_windows=str(REF/'01_candidate_windows.csv'),
    checkpoint=str(MODEL_SOURCE),total_circles=34,
    development_circles=sorted(DEV),independent_holdouts=sorted(HOLDOUT),
    model='M2 native TinyUNet, trained in NB02C; NO training here',
    validation_policy='7 independent holdouts; 27 development circles apparent fit only',
    full_34circle_validation='NOT DONE: requires three spatial CV folds',
    historical_cv_untouched=str(ROOT/'Outputs/V3/02D_spatial_cv'),
    files=['02D_quick34_circle_predictions.csv','02D_quick34_overview.csv',
           '02D_quick34_site_summary.csv','02D_quick34_development_vs_holdout.png'])
(OUT/'02D_quick34_contract.json').write_text(json.dumps(contract,indent=2))
print('Quick34 finished. Full 34-circle CV remains the next step if needed.')
